# Objective 1 (E1/E2) — Authoritative Streaming Execution Engine

**Thesis:** *Adaptive Low-Latency Fraud Detection in Streaming Financial Systems with LLM-Augmented Explainability*  
**Experiment:** Objective 1: Policy Comparison under Natural Financial Stream (E1: Baseline vs. Adaptive; E2: Full Factorial Grid)  
**Methodology:** Prequential Evaluation (Test-Then-Train), Deterministic State Machine Benchmark, Atomic Checkpointing & Resumable Manifest  


## 1. Import Required Libraries and Environment Setup


In [1]:
import os
import sys
import time
import json
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import river
import sklearn

print(f"Python environment: {sys.executable}")
print(f"NumPy version:     {np.__version__}")
print(f"Pandas version:    {pd.__version__}")
print(f"River version:     {river.__version__}")
print(f"Scikit-learn:      {sklearn.__version__}")


Python environment: C:\Projects\Thesis\.venv\Scripts\python.exe
NumPy version:     2.5.1
Pandas version:    3.0.3
River version:     0.26.1
Scikit-learn:      1.9.0


## 2. Execution Mode Selection & Job Configuration


In [2]:
# TOP-LEVEL EXECUTION MODE SELECTION
# Options:
#   "SMOKE" : Local verification mode (fast subset, 2 reproducibility seeds)
#   "FULL"  : Authoritative Kaggle/Production execution (590,540 rows)
RUN_MODE = "SMOKE"

# JOB SELECTION
# On Kaggle, you can run all 4 policies or filter specific policies/seeds
POLICIES = ["P0", "P1", "P2", "P3"]

if RUN_MODE == "SMOKE":
    SEEDS = [42, 101]  # 2 seeds to verify determinism/reproducibility
    SMOKE_ROW_LIMIT = 5000
    WARMUP_FRAC = 0.15
    WINDOW_SIZE = 500
    N_INTERVAL = 1000
    GRACE_PERIOD = 50
    DELTA = 0.002
    ROLLING_WINDOW_SIZE = 500
    MAX_TRAJECTORY_POINTS = 100
else:
    SEEDS = [42, 101]  # Invariant reproducibility checks (all deterministic)
    SMOKE_ROW_LIMIT = None
    WARMUP_FRAC = 0.15
    WINDOW_SIZE = 5000
    N_INTERVAL = 10000
    GRACE_PERIOD = 200
    DELTA = 0.002
    ROLLING_WINDOW_SIZE = 5000
    MAX_TRAJECTORY_POINTS = 200

print(f"Configured RUN_MODE:      {RUN_MODE}")
print(f"Target Policies:          {POLICIES}")
print(f"Target Seeds:             {SEEDS}")
print(f"Total Jobs to Process:    {len(POLICIES) * len(SEEDS)}")


Configured RUN_MODE:      SMOKE
Target Policies:          ['P0', 'P1', 'P2', 'P3']
Target Seeds:             [42, 101]
Total Jobs to Process:    8


## 3. Project Root & Directory Hierarchy Configuration


In [3]:
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for parent in [current] + list(current.parents):
        if (parent / 'src').is_dir() and ((parent / 'docs').is_dir() or (parent / 'pyproject.toml').is_file()):
            return parent
    return current

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root resolved: {PROJECT_ROOT}")

# Outputs directory hierarchy
CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints" / "objective1_runs"
MANIFEST_PATH = PROJECT_ROOT / "outputs" / "checkpoints" / "objective1_manifest.json"
ZIP_ARCHIVE_PATH = PROJECT_ROOT / "outputs" / "objective1_artifacts_live.zip"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint directory:   {CHECKPOINT_DIR}")
print(f"Manifest path:          {MANIFEST_PATH}")
print(f"Live archive mirror:    {ZIP_ARCHIVE_PATH}")


Project root resolved: C:\Projects\Thesis
Checkpoint directory:   C:\Projects\Thesis\outputs\checkpoints\objective1_runs
Manifest path:          C:\Projects\Thesis\outputs\checkpoints\objective1_manifest.json
Live archive mirror:    C:\Projects\Thesis\outputs\objective1_artifacts_live.zip


## 4. Benchmark Dataset Discovery & Ingestion


In [4]:
def resolve_ieee_cis_dir() -> Path:
    candidates = [
        Path("/kaggle/input/ieee-fraud-detection"),
        Path("/kaggle/input/ieee-cis-fraud-detection"),
        Path("/kaggle/working/datasets/ieee_cis"),
        PROJECT_ROOT / "datasets" / "ieee_cis",
        PROJECT_ROOT / "data" / "ieee_cis",
    ]
    for c in candidates:
        if c.exists() and (c / 'train_transaction.csv').exists():
            return c
    raise FileNotFoundError(f"Could not locate IEEE-CIS dataset in candidates: {candidates}")

IEEE_CIS_DIR = resolve_ieee_cis_dir()
print(f"IEEE-CIS data source resolved at: {IEEE_CIS_DIR}")

from src.data.loader import load_ieee_cis
from src.utils.seed import set_seed

df_raw = load_ieee_cis(data_dir=IEEE_CIS_DIR, split="train", join_identity=True)
if SMOKE_ROW_LIMIT is not None:
    df_raw = df_raw.iloc[:SMOKE_ROW_LIMIT].copy()
print(f"Ingested raw transactions: {len(df_raw):,} rows, {len(df_raw.columns)} columns")


IEEE-CIS data source resolved at: C:\Projects\Thesis\datasets\ieee_cis
Ingested raw transactions: 5,000 rows, 434 columns


## 5. Chronological Verification & Warmup-Only Preprocessing


In [5]:
# Verify strict chronological monotonicity of TransactionDT
dt_series = df_raw['TransactionDT']
is_monotonic = bool(dt_series.is_monotonic_increasing)
print(f"Chronological Monotonicity Check (TransactionDT): {is_monotonic}")
assert is_monotonic, "Dataset MUST be sorted monotonically by TransactionDT!"

# Compute dataset fingerprint for auditability
data_fingerprint = hashlib.sha256(
    f"{len(df_raw)}_{df_raw['TransactionID'].iloc[0]}_{df_raw['TransactionID'].iloc[-1]}".encode('utf-8')
).hexdigest()[:16]
print(f"Dataset fingerprint: {data_fingerprint}")

# Compute warmup split
n_total = len(df_raw)
warmup_size = int(n_total * WARMUP_FRAC)
stream_size = n_total - warmup_size
print(f"Warmup size (Phase 1): {warmup_size:,} ({WARMUP_FRAC*100:.1f}%)")
print(f"Stream size (Phase 2): {stream_size:,} ({(1-WARMUP_FRAC)*100:.1f}%)")

# Preprocessing: Fit ONLY on warmup data to strictly prevent future leakage
from src.data.preprocessing import StreamingPreprocessor

preprocessor = StreamingPreprocessor(
    dataset_name="ieee_cis",
    scale_features=False,
)

print("Fitting streaming preprocessor on warmup partition only...")
preprocessor.fit(df_raw.iloc[:warmup_size])
print(f"Preprocessor fitted. Total features after transform: {len(preprocessor.feature_names):,}")

print("Transforming full dataset...")
X_all, y_all, seg_all = preprocessor.transform(df_raw)

print(f"Feature matrix X: {X_all.shape}")
print(f"Target Series y:  {len(y_all):,} (fraud rate = {y_all.mean():.4f})")
print(f"Segment Series:   {seg_all.nunique() if seg_all is not None else 0} distinct segments")


Chronological Monotonicity Check (TransactionDT): True
Dataset fingerprint: 560d5218a195a6e6
Warmup size (Phase 1): 750 (15.0%)
Stream size (Phase 2): 4,250 (85.0%)
Fitting streaming preprocessor on warmup partition only...
Preprocessor fitted. Total features after transform: 431
Transforming full dataset...
Feature matrix X: (5000, 431)
Target Series y:  5,000 (fraud rate = 0.0218)
Segment Series:   5 distinct segments


## 6. Manifest Initialization & Resumption Discovery


In [6]:
from src.pipeline.manifest import ExperimentManifest

manifest = ExperimentManifest(
    manifest_path=MANIFEST_PATH,
    artifacts_dir=CHECKPOINT_DIR,
    zip_archive_path=ZIP_ARCHIVE_PATH,
    experiment_name="Objective1_E1_E2",
    dataset_name="IEEE-CIS",
)

print(f"Loaded manifest with {len(manifest.jobs)} known jobs.")
completed_count = sum(1 for j in manifest.jobs.values() if j.status == 'COMPLETED')
print(f"Completed jobs on disk: {completed_count}")


Loaded manifest with 8 known jobs.
Completed jobs on disk: 8


## 7. Trajectory Downsampling Helper


In [7]:
from sklearn.metrics import precision_recall_curve, auc

def extract_downsampled_trajectories(records: List[Any], window_size: int = 5000, max_points: int = 150) -> Dict[str, Any]:
    """Extract rolling PR-AUC trajectory downsampled to a compact representation for serialization."""
    n_records = len(records)
    if n_records == 0:
        return {"indices": [], "rolling_pr_auc": []}

    step = max(1, n_records // max_points)
    indices = []
    pr_aucs = []

    y_true_all = [r.y_true for r in records]
    y_prob_all = [r.y_prob for r in records]

    for end_idx in range(window_size, n_records + 1, step):
        start_idx = end_idx - window_size
        y_w = y_true_all[start_idx:end_idx]
        p_w = y_prob_all[start_idx:end_idx]

        if sum(y_w) > 0 and len(y_w) - sum(y_w) > 0:
            prec, rec, _ = precision_recall_curve(y_w, p_w)
            score = float(auc(rec, prec))
        else:
            score = 0.0

        indices.append(end_idx)
        pr_aucs.append(round(score, 5))

    return {
        "indices": indices,
        "rolling_pr_auc": pr_aucs,
        "window_size": window_size,
    }

print("Trajectory downsampling helper defined.")


Trajectory downsampling helper defined.


## 8. Authoritative Job Execution Loop (Resumable & Atomic)


In [8]:
from src.pipeline.runner import PrequentialRunner

total_jobs = len(POLICIES) * len(SEEDS)
job_idx = 0

print(f"Starting execution loop over {total_jobs} potential jobs...")

config_snapshot = {
    "warmup_size": warmup_size,
    "stream_size": stream_size,
    "warmup_frac": WARMUP_FRAC,
    "window_size": WINDOW_SIZE,
    "n_interval": N_INTERVAL,
    "grace_period": GRACE_PERIOD,
    "delta": DELTA,
    "run_mode": RUN_MODE,
}

for policy in POLICIES:
    for seed in SEEDS:
        job_idx += 1
        job_id = manifest.make_job_id(policy, seed)

        # Check if already completed
        if manifest.is_job_completed(policy, seed):
            print(f"[{job_idx}/{total_jobs}] Job {job_id} already COMPLETED on disk -> SKIPPING.")
            continue

        print(f"[{job_idx}/{total_jobs}] Executing Job {job_id}...")
        manifest.record_start(policy, seed, dataset_fingerprint=data_fingerprint)
        set_seed(seed)

        t_start = time.perf_counter()
        try:
            runner = PrequentialRunner.from_config(
                policy_str=policy,
                detector_str="adwin",
                grace_period=GRACE_PERIOD,
                delta=DELTA,
                window_size=WINDOW_SIZE,
                n_interval=N_INTERVAL,
                detector_kwargs={"delta": DELTA},
                segment_aware=(policy == "P3"),
                seed=seed,
                p0_mode="incremental",
            )

            # Run prequential stream
            run_result = runner.run(
                X=X_all,
                y=y_all,
                segment=seg_all,
                warmup_size=warmup_size,
            )

            wall_clock_s = time.perf_counter() - t_start

            # Downsample trajectories for compact storage
            trajectories = extract_downsampled_trajectories(
                run_result.records,
                window_size=ROLLING_WINDOW_SIZE,
                max_points=MAX_TRAJECTORY_POINTS,
            )

            # Persist artifact atomically
            artifact_path = manifest.record_completion(
                policy=policy,
                seed=seed,
                run_result=run_result,
                wall_clock_duration_s=wall_clock_s,
                dataset_fingerprint=data_fingerprint,
                code_version="authoritative_o1_v2",
                config_snapshot=config_snapshot,
                trajectories=trajectories,
            )

            fm = run_result.final_metrics
            lp = run_result.latency_percentiles
            pr_val = fm.pr_auc if hasattr(fm, "pr_auc") else fm.get("pr_auc", 0.0)
            roc_val = fm.roc_auc if hasattr(fm, "roc_auc") else fm.get("roc_auc", 0.0)
            p95_val = lp.get("p95_ms", lp.get("p95", 0.0) * 1000)
            print(
                f"    [COMPLETED] Duration: {wall_clock_s:.1f}s | "
                f"PR-AUC: {pr_val:.4f} | ROC-AUC: {roc_val:.4f} | "
                f"Adaptations: {len(run_result.adaptation_log)} | "
                f"Latency p95: {p95_val:.2f}ms | Saved: {artifact_path.name}"
            )


        except Exception as ex:
            manifest.record_failure(policy, seed, str(ex))
            print(f"    [FAILED] Job {job_id} encountered error: {ex}")
            raise

print("Execution loop complete. All target jobs are up to date.")


Starting execution loop over 8 potential jobs...
[1/8] Job IEEE-CIS_Objective1_E1_E2_P0_seed42 already COMPLETED on disk -> SKIPPING.
[2/8] Job IEEE-CIS_Objective1_E1_E2_P0_seed101 already COMPLETED on disk -> SKIPPING.
[3/8] Job IEEE-CIS_Objective1_E1_E2_P1_seed42 already COMPLETED on disk -> SKIPPING.
[4/8] Job IEEE-CIS_Objective1_E1_E2_P1_seed101 already COMPLETED on disk -> SKIPPING.
[5/8] Job IEEE-CIS_Objective1_E1_E2_P2_seed42 already COMPLETED on disk -> SKIPPING.
[6/8] Job IEEE-CIS_Objective1_E1_E2_P2_seed101 already COMPLETED on disk -> SKIPPING.
[7/8] Job IEEE-CIS_Objective1_E1_E2_P3_seed42 already COMPLETED on disk -> SKIPPING.
[8/8] Job IEEE-CIS_Objective1_E1_E2_P3_seed101 already COMPLETED on disk -> SKIPPING.
Execution loop complete. All target jobs are up to date.


## 9. Execution Summary Table & Checkpoint Verification


In [9]:
summary_df = manifest.get_summary_dataframe()
display_cols = [
    'job_id', 'policy', 'seed', 'status', 'pr_auc', 'roc_auc',
    'adaptation_count', 'latency_p95_ms', 'wall_clock_duration_s'
]
available_cols = [c for c in display_cols if c in summary_df.columns]
print("\nObjective 1 Execution Summary:")
print(summary_df[available_cols].to_string(index=False))

# Verify that all target jobs are marked COMPLETED
for policy in POLICIES:
    for seed in SEEDS:
        assert manifest.is_job_completed(policy, seed), f"Job {policy}_seed{seed} is NOT marked COMPLETED!"

print("\n[INVARIANT PASSED] All target jobs successfully executed, verified, and persisted atomically.")
print(f"Artifacts ready for analysis in: {CHECKPOINT_DIR}")
print(f"Live archive ready for download:  {ZIP_ARCHIVE_PATH}")



Objective 1 Execution Summary:
                              job_id policy  seed    status   pr_auc  roc_auc  adaptation_count  latency_p95_ms  wall_clock_duration_s
 IEEE-CIS_Objective1_E1_E2_P0_seed42     P0    42 COMPLETED 0.070838 0.558681                 0             NaN                  9.426
IEEE-CIS_Objective1_E1_E2_P0_seed101     P0   101 COMPLETED 0.070838 0.558681                 0          0.0313                  9.869
 IEEE-CIS_Objective1_E1_E2_P1_seed42     P1    42 COMPLETED 0.024105 0.492583                 4          0.0234                 11.799
IEEE-CIS_Objective1_E1_E2_P1_seed101     P1   101 COMPLETED 0.024105 0.492583                 4          0.0224                 12.097
 IEEE-CIS_Objective1_E1_E2_P2_seed42     P2    42 COMPLETED 0.070838 0.558681                 0          0.0267                  8.858
IEEE-CIS_Objective1_E1_E2_P2_seed101     P2   101 COMPLETED 0.070838 0.558681                 0          0.0270                  8.691
 IEEE-CIS_Objective1_E1